# Build and simulate circuits in Python

## Prerequisites

* Python 3.10+ in a virtual environment with **this branch's built `qdk` package**, `ipykernel`, and IPython installed. The released package may not have this experimental API.
* JupyterLab, or VS Code with its Python and Jupyter extensions. Select the Python kernel containing that QDK build.
* For the optional GPU cell: a GPU and drivers supported by the QDK GPU simulator. No GPU is required for the default run.

See the [sample setup guide](README.md) for build and execution instructions. No Q#, C, Go, or Fortran kernel is needed.

The circuit API constructs and compiles programs; it does not have a circuit `run` method. We compile to QIR and pass that program to `qdk.simulation.run_qir`. A pass pipeline's `run` method transforms a program; it does not simulate it.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import math

from qdk import Result, TargetProfile
from qdk.circuit import (
    ClassicalRegister,
    PassPipeline,
    QuantumCircuit,
    QuantumGraph,
    QuantumRegister,
)
from qdk.simulation import run_qir

SHOTS = 128
RUN_GPU = False  # Opt in; GPU errors never fall back to CPU.


def correlated_counts(samples):
    assert len(samples) == SHOTS
    assert all(
        len(shot) == 2 and shot[0] == shot[1] and shot[0] in (Result.Zero, Result.One)
        for shot in samples
    )
    return Counter(
        "".join("1" if bit == Result.One else "0" for bit in shot) for shot in samples
    )

## A Base-profile Bell circuit

Named registers are views over qubit and classical-bit indices. H creates a superposition, CX entangles the qubits, and terminal measurements produce only `00` or `11`. The counts are statistical, not necessarily exactly half of each outcome.

In [ ]:
qubits = QuantumRegister("bell", 2)
readout = ClassicalRegister("readout", 2)
base = QuantumCircuit.from_registers(qubits, readout)
base.h(qubits[0]).cx(qubits[0], qubits[1])
base.measure(qubits[0], readout[0]).measure(qubits[1], readout[1])
print(base)
print([(instruction.name, instruction.qubits) for instruction in base])

base_qir = base.to_qir(TargetProfile.Base)
base_samples = run_qir(base_qir, shots=SHOTS, seed=42, type="cpu")
print("CPU Bell counts:", correlated_counts(base_samples))

## Adaptive measurement feedback

Measure qubit 0, then apply X to qubit 1 only when the measurement is One. This produces the same two possible bitstrings as the Bell circuit, but through classical feedback rather than entanglement. Base cannot express this runtime branch; the compiler reports the missing capability instead of dropping the condition.

In [ ]:
adaptive = QuantumCircuit(2, 2)
adaptive.h(0).measure(0, 0)
with adaptive.if_test(0, 1):
    adaptive.x(1)
adaptive.measure(1, 1)
print(adaptive)

try:
    adaptive.to_qir(TargetProfile.Base)
except ValueError as error:
    assert "Adaptive" in str(error)
    print("Expected Base refusal:", error)
else:
    raise AssertionError("Base must reject measurement feedback")

adaptive_qir = adaptive.to_qir(TargetProfile.Adaptive)
adaptive_samples = run_qir(adaptive_qir, shots=SHOTS, seed=42, type="cpu")
print("CPU feedback counts:", correlated_counts(adaptive_samples))

## Request the GPU explicitly

Set `RUN_GPU = True` in the setup cell and rerun from there. Both QIR programs run on the selected GPU backend. GPU initialization errors remain visible; there is no CPU fallback. A two-qubit program demonstrates the interface, not a GPU performance advantage. CPU and GPU samples need not match shot-for-shot, even with the same seed.

In [ ]:
if RUN_GPU:
    for name, qir in [("Bell", base_qir), ("feedback", adaptive_qir)]:
        samples = run_qir(qir, shots=SHOTS, seed=42, type="gpu")
        print(f"GPU {name} counts:", correlated_counts(samples))
else:
    print("GPU execution disabled; set RUN_GPU = True to enable it.")

## Exact snapshots and interchange

`to_bytes` produces the typed binary `.qdk` snapshot used by all four language samples. Always use binary file I/O. It preserves circuit structure, registers, and symbolic parameters; it is not JSON or a HUGR document. `to_text` produces a readable typed S-expression, saved alongside it as a UTF-8 `.sexpr` file. OpenQASM is useful for interchange, but is not an exact snapshot format.

The cell prints the full paths of both snapshot formats. Files remain available until the final cleanup cell; copy any you want to retain outside the temporary directory first.

In [ ]:
work = TemporaryDirectory(prefix="qdk-python-circuits-")
directory = Path(work.name).resolve()
print("Artifact directory:", directory)
for name, circuit in [("base", base), ("adaptive", adaptive)]:
    binary_path = directory / f"{name}.qdk"
    text_path = directory / f"{name}.sexpr"
    binary_path.write_bytes(circuit.to_bytes())
    text_path.write_text(circuit.to_text(), encoding="utf-8")
    loaded = QuantumCircuit.from_bytes(binary_path.read_bytes())
    loaded_text = QuantumCircuit.from_text(text_path.read_text(encoding="utf-8"))
    assert loaded.to_bytes() == loaded_text.to_bytes() == circuit.to_bytes()
    for path in [binary_path, text_path]:
        print(f"{path} ({path.stat().st_size} bytes)")

print("Register names:", [register.name for register in base.quantum_registers])
print("S-expression preview:", base.to_text()[:250])
print(base.to_qasm())

## Parameter sweeps without rebuilding the template

Parameters are symbolic inputs bound before compilation, not runtime classical variables. Binding produces a new circuit. An Rx rotation by zero always measures Zero; Rx by pi always measures One. Intermediate angles produce a distribution. Parameter arithmetic is resolved during binding.

In [ ]:
template = QuantumCircuit(1, 1)
theta = template.parameter("theta")
template.rx(theta / 2, 0).measure(0, 0)

sweep = {}
for value in [0.0, math.pi, 2 * math.pi]:
    bound = template.assign_parameters([(theta, value)])
    assert bound.parameters == []
    samples = run_qir(
        bound.to_qir(TargetProfile.Base), shots=SHOTS, seed=42, type="cpu"
    )
    assert len(samples) == SHOTS
    assert all(
        len(shot) == 1 and shot[0] in (Result.Zero, Result.One) for shot in samples
    )
    sweep[value] = samples
    print(f"theta={value:.3f}:", Counter(str(shot[0]) for shot in samples))

assert all(shot == [Result.Zero] for shot in sweep[0.0])
assert all(shot == [Result.One] for shot in sweep[2 * math.pi])
assert template.parameters == [theta]

## Inspect the graph and run compiler passes

Lowering exposes the quantum dataflow graph, including allocation and capability flow that a circuit diagram hides. Graph snapshots retain that representation. The default pass pipeline can cancel an adjacent H/H pair without changing the observable result. The original circuit remains unchanged.

In [ ]:
graph = base.to_graph()
assert graph.diagnostics() == []
print("Graph operation counts:", Counter(node.operation for node in graph.nodes))
restored_graph = QuantumGraph.from_bytes(graph.to_bytes())
assert restored_graph.diagnostics() == []
assert restored_graph.to_bytes() == graph.to_bytes()

redundant = QuantumCircuit(1, 1)
redundant.h(0).h(0).x(0).measure(0, 0)
optimized = PassPipeline().run(redundant)
assert optimized.circuit is not None
assert [instruction.name for instruction in redundant] == ["h", "h", "x", "measure_z"]
assert [instruction.name for instruction in optimized.circuit] == ["x", "measure_z"]
print("Passes that changed the program:", optimized.changed)
print(optimized.circuit)
original_samples = run_qir(
    redundant.to_qir(TargetProfile.Base), shots=SHOTS, type="cpu"
)
optimized_samples = run_qir(
    optimized.circuit.to_qir(TargetProfile.Base), shots=SHOTS, type="cpu"
)
assert original_samples == optimized_samples == [[Result.One]] * SHOTS

## Next experiments

Continue with [Circuit API experiments and custom graph passes](circuit_api_python_advanced.ipynb) to build a longer entangled register, add an else body to the feedback circuit, save a parameterized template for a separate process to bind, and write graph rewrites and expansions. The [C](circuit_api_c.ipynb), [Go](circuit_api_go.ipynb), and [Fortran](circuit_api_fortran.ipynb) notebooks build circuits outside Python and load the same binary snapshot format. More API examples are in the [circuit guide](../../../source/qdk_package/CIRCUIT-USAGE.md).

## Cleanup

Copy any `.qdk` or `.sexpr` files you want to keep out of `directory` before running this cell. It removes this notebook's temporary snapshot files.

In [ ]:
work.cleanup()
assert not directory.exists()